In [0]:
from bs4 import BeautifulSoup
from bs4 import NavigableString
import re
from datetime import datetime

class HTMLDocumentParser():

    def parse(self, raw_document):

        soup = self._create_soup(raw_document)

        self._remove_noise(soup)

        self._remove_front_matter(soup)

        title = self._extract_title(soup)

        tables=soup.find_all("table")
        
        for table in tables:
            
            table_rows = self._extract_table(table)
            markdown = self._table_to_markdown(table_rows)
            table.replace_with(
                NavigableString("\n" + markdown + "\n")
            )
        clean_text = self._extract_clean_text(soup)

        clean_text = self._normalize_text(clean_text)
        
        sections = self._extract_sections(clean_text)
        
        return sections


    def _remove_front_matter(self, soup):
        toc = soup.find(
            string=lambda text: text and "TABLE OF CONTENTS" in text.upper()
        )

        if not toc:
            return

        toc_element = toc.parent

        first_item_link = toc_element.find_next(
            "a",
            string=lambda text: text and re.match(
                r"Item\s+1[A-Z]?\.",
                text.strip(),
                re.IGNORECASE
            )
        )
        

        if not first_item_link:
            return

        anchor_id = first_item_link.get("href")
        
        if not anchor_id or not anchor_id.startswith("#"):
            return

        anchor_id = anchor_id[1:]

        actual_start = soup.find(id=anchor_id)
        
        if not actual_start:
            return

        body = soup.body

        for element in list(body.children):

            if element == actual_start:
                break

            element.extract()
    def _extract_table(self,table):
        rows = []

        for tr in table.find_all("tr", recursive=False):

            row = []

            cells = tr.find_all(["th", "td"], recursive=False)

            for cell in cells:

                value = cell.get_text(" ", strip=True)

                row.append(value)

            if row:
                rows.append(row)

        return rows
    
    def _table_to_markdown(self,rows):
        if not rows:
            return ""

        markdown = []

        # Header
        header = "| " + " | ".join(rows[0]) + " |"
        markdown.append(header)

        # Separator
        separator = "| " + " | ".join(["---"] * len(rows[0])) + " |"
        markdown.append(separator)

        # Data rows
        for row in rows[1:]:

            line = "| " + " | ".join(row) + " |"

            markdown.append(line)

        return "\n".join(markdown)

    def _create_soup(self, raw_document):
        return BeautifulSoup(raw_document, 'html.parser')
    
    def _remove_noise(self, soup):
       for tag in soup.find_all([
        "script",
        "style",
        "noscript",
        "svg",
        "canvas"
        ]):
            tag.decompose()
    
    def _extract_title(self, soup):
        title = soup.find("title")
        if title:
            return title.get_text(" ", strip=True)
        return None
    
    def _extract_clean_text(self, soup):
        return soup.get_text(separator="\n",
        strip=True)
    
    def _normalize_text(self, text):

        lines = []

        for line in text.splitlines():

            line = re.sub(r'\s+', ' ', line)
            line = line.strip()

            if line:
                lines.append(line)

        return "\n".join(lines)

    def _is_section_heading(self, text):

        pattern = r"^Item\s+\d+[A-Z]?\."

        return bool(re.match(pattern, text, re.IGNORECASE))

    def _extract_sections(self, text):
        sections = []
        current_section = None
        content = []
        section_num=1
        for line in text.splitlines():
            
            line = line.strip()

            if not line:
                continue

            if self._is_section_heading(line):

                if current_section:

                    current_section["content"] = "\n".join(content)

                    sections.append(current_section)
                print(section_num)
                current_section = {
                    "heading_level": 1,
                    "number":section_num,
                    "heading_text": line,
                    "content": None
                }
                section_num+=1
                content = []

            elif current_section:

                content.append(line)

        if current_section:

            current_section["content"] = "\n".join(content)

            sections.append(current_section)

        return sections

    def _build_document(self, title, text):
        return Document(
            title=title,
            text=text,
            created_at=datetime.now()
        )

In [0]:
from pyspark.sql.types import *

silver_schema = StructType([
    StructField("company_name", StringType(), True),
    StructField("cik", StringType(), True),
    StructField("filling_date", StringType(), True),
    StructField("accession_number", StringType(), True),
    StructField("filling_type", StringType(), True),
    StructField("doc_id",StringType(),True),
    StructField("sec_id",StringType(),True),
    StructField("sec_text",StringType(),True),
    StructField("processing_timestamp",StringType(),True),
    StructField("sec_number",StringType(),True),
    StructField("sec_title",StringType(),True),
    StructField("file_name",StringType(),True)
])

In [0]:
from pyspark.sql import SparkSession
import time


path="workspace.rag.bronze_delta"
silver_table="rag.silver_delta"
bronze_table="rag.bronze_delta"

#we will read bronze table and then read the latest data by comparing 
spark= SparkSession.builder.appName("Silver").getOrCreate()










df=spark.table(bronze_table)



if spark.catalog.tableExists(silver_table):
    #get new data from bronze and then compare with silver table date and MERGE
    pass
else:
    #for now we are just taking whole data to check how our spark table performs , but later we will take only latest data
    silver_data=[]
    for row in df.collect():
        
        html=row['raw_data']
        parser=HTMLDocumentParser()
        data=parser.parse(html)
        
        for section in data:
            silver_row={}
            silver_row["sec_text"]=section["content"]
            silver_row["sec_title"]=section["heading_text"]
            silver_row["sec_number"]=section["number"]
            silver_row["sec_id"]=row["doc_id"]+"_"+silver_row["sec_title"]+"_"+str(silver_row["sec_number"])
            silver_row["company_name"]=row["company_name"]
            silver_row["cik"]=row["cik"]
            silver_row["filling_date"]=row["filling_date"]
            silver_row["accession_number"]=row["accession_number"]
            silver_row["filling_type"]=row["filling_type"]
            silver_row["processing_timestamp"]=time.time_ns() // 1_000_000
            silver_row["doc_id"]=row["doc_id"]
            silver_data.append(silver_row)
            
    if len(silver_data)>0:
        spark.createDataFrame(silver_data,schema=silver_schema).write.mode("append").saveAsTable(silver_table)

    